<a href="https://colab.research.google.com/github/zzzer0-wav/myDTA_2026/blob/main/ML/logreg_pipeline_10TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [15]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [16]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [17]:
# TODO: виведи баланс класів
display(clients['upgraded'].value_counts(normalize=True))

display(clients.info())

,proportion
upgraded,
0,0.515556
1,0.484444


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   age       900 non-null    int64 
 1   tenure    900 non-null    int64 
 2   usage     900 non-null    int64 
 3   support   900 non-null    int64 
 4   plan      900 non-null    object
 5   region    900 non-null    object
 6   upgraded  900 non-null    int64 
dtypes: int64(5), object(2)
memory usage: 49.3+ KB


None

✍️ Випиши списки стовпців (знадобляться далі):
> числові: 5
> категорійні: 2

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [18]:
clients.columns

Index(['age', 'tenure', 'usage', 'support', 'plan', 'region', 'upgraded'], dtype='object')

In [19]:
from sklearn.model_selection import train_test_split

# TODO: створи X, y та зроби поділ
X = clients[['age', 'tenure', 'usage', 'support', 'plan', 'region']]
y = clients['upgraded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess
num_cols = ['age', 'tenure', 'usage', 'support']
cat_cols = ['plan', 'region']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe
pipe = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [51]:
# TODO: навчи pipe і виведи accuracy на тесті
pipe.fit(X_train, y_train)
print(f'Accuracy on test: {round(pipe.score(X_test, y_test), 3)}')

Accuracy on test: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [23]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: передбач, виведи матрицю плутанини та звіт
pred = pipe.predict(X_test)

cm = confusion_matrix(y_test, pred)
print(f'Confusion Matrix:\n{cm}')

print(f'\nClassification report: \n{classification_report(y_test, pred)}')

Confusion Matrix:
[[80 13]
 [15 72]]

Classification report: 
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [52]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC
proba = pipe.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, proba)
print(f'ROC-AUC = {roc.round(2)}')
print(f'Приклад ймовірності перших 5 клієнтів: {proba[:5].round(2)}')

ROC-AUC = 0.93
Приклад ймовірності перших 5 клієнтів: [0.07 0.15 0.   0.97 0.19]


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [25]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|
names = pipe.named_steps['prep'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'features':names,
    'coef':coefs
}).sort_values('coef', key=abs, ascending=False)

print()
coef_df

,features,coef
2,num__usage,2.027135
1,num__tenure,1.648386
6,cat__plan_сімейний,1.411953
4,cat__plan_базовий,-1.339728
3,num__support,-0.835947
7,cat__region_захід,0.537637
0,num__age,-0.406875
10,cat__region_схід,-0.340620
8,cat__region_південь,-0.189350
9,cat__region_північ,0.136239


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум активність користувача (usage 2.03) - чим більше часу людина проводить на платформі, тим більше шанс, що вона купить преміум. Також позитивно впливають на це те, як довго клієнт з нами (tenure 1.64) та сімейний план (1.41).
Найбільше знижує шанс - базовий тариф (plan базовий -1.33) та кількість звернень у підтримку (support -0.83) - чим більше скаржиться, тим менше заохочення платити більше.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [26]:
clients.columns

Index(['age', 'tenure', 'usage', 'support', 'plan', 'region', 'upgraded'], dtype='object')

In [34]:
# TODO: створи new_client, виведи рішення та ймовірність переходу
client1 = pd.DataFrame ([{
    'age':30,
    'tenure':24,
    'usage':120,
    'support':0,
    'plan':'сімейний',
    'region':'захід'
}])

print(client1)

print('\nКлієнт -', 'апгрейднеться' if pipe.predict(client1) else 'не буде витрачати кошти')

   age  tenure  usage  support      plan region
0   30      24    120        0  сімейний  захід

Клієнт - апгрейднеться


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [37]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид
cv = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
print(f'CV roc_auc: {np.round(cv, 3)} | середнє: {cv.mean():.3f} ± {cv.std():.3f}')

CV roc_auc: [0.934 0.931 0.94  0.891 0.921] | середнє: 0.923 ± 0.018


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [64]:
# Місце для бонусних експериментів
preprocess2 = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

pipe2 = Pipeline([
    ('prep',preprocess2),
    ('model', LogisticRegression(max_iter=1000))
])

pipe2.fit(X_train, y_train)

print(f'Accuracy: {round(pipe2.score(X_test, y_test), 3)}')

print('\nROC-AUC значення з масштабуванням:')
proba = pipe.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, proba)
print(f'ROC-AUC = {roc.round(2)}')
print(f'Приклад ймовірності перших 5 клієнтів: {proba[:5].round(2)}')

print('\nROC-AUC значення без масштабування цифрових значень:')

proba2 = pipe2.predict_proba(X_test)[:, 1]
roc2 = roc_auc_score(y_test, proba2)
print(f'ROC-AUC = {roc2.round(2)}')
print(f'Приклад ймовірності перших 5 клієнтів: {proba2[:5].round(2)}')

#______________________________________________________

preprocess3 = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

pipe3 = Pipeline([
    ('prep',preprocess3),
    ('model',LogisticRegression(max_iter=20))
])

pipe3.fit(X_train, y_train)
print('\n____________________')

print('\nROC-AUC значення з масштабуванням: (max_iter=20)')
proba3 = pipe3.predict_proba(X_test)[:, 1]
roc3 = roc_auc_score(y_test, proba3)
print(f'ROC-AUC = {roc3.round(2)}')
print(f'Приклад ймовірності перших 5 клієнтів: {proba3[:5].round(2)}')


preprocess4 = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

pipe4 = Pipeline([
    ('prep',preprocess4),
    ('model',LogisticRegression(max_iter=20))
])

pipe4.fit(X_train, y_train)
print('\n____________________')

print('\nROC-AUC значення без масштабування: (max_iter=20)')
proba4 = pipe4.predict_proba(X_test)[:, 1]
roc4 = roc_auc_score(y_test, proba4)
print(f'ROC-AUC = {roc4.round(2)}')
print(f'Приклад ймовірності перших 5 клієнтів: {proba4[:5].round(2)}')

Accuracy: 0.844

ROC-AUC значення з масштабуванням:
ROC-AUC = 0.93
Приклад ймовірності перших 5 клієнтів: [0.07 0.15 0.   0.97 0.19]

ROC-AUC значення без масштабування цифрових значень:
ROC-AUC = 0.93
Приклад ймовірності перших 5 клієнтів: [0.06 0.14 0.   0.97 0.18]

____________________

ROC-AUC значення з масштабуванням: (max_iter=20)
ROC-AUC = 0.93
Приклад ймовірності перших 5 клієнтів: [0.07 0.15 0.   0.97 0.19]

____________________

ROC-AUC значення без масштабування: (max_iter=20)
ROC-AUC = 0.87
Приклад ймовірності перших 5 клієнтів: [0.03 0.23 0.01 0.9  0.79]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Результат експеременту: якщо не масштабувати цифрові значення, то різниця результатів для цих даних майже зовсім не відрізняється, але це не означає, що масштабування не потрібне, просто у перших двох випадках ми надали моделі 1000 шагів для навчання та покращення результату, та вона впоралася менш ніж за 1000 корегувань. А у другому прикладі я надав моделі по 20 кроків для навчання. Модель з масшатабуванням впоралася так само як модель з та без масштабування з 1000 кроків, а без масштабування модель не встигла навчитися за 20 кроків, видала хибкі резульати та попередження, що цього їй ще замало.

Масштабування не впливає на якість моделі, але впливає на її швидкість навчання та стабільність.

In [68]:
preprocess_bal = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

pipe_bal = Pipeline([
    ('prep', preprocess_bal),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipe_bal.fit(X_train, y_train)

from sklearn.metrics import confusion_matrix, classification_report

y_pred = pipe.predict(X_test)
y_pred_bal = pipe_bal.predict(X_test)

print('=== БЕЗ balanced ===')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print('=== З balanced ===')
print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

=== БЕЗ balanced ===
[[80 13]
 [15 72]]
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180

=== З balanced ===
[[80 13]
 [15 72]]
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



Класи у датасеті (0 та 1) сбалансовані (~50/50), тому результат с balanced такий самий як і без нього.

In [69]:
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")

Поріг 0.5: матриця
[[80 13]
 [15 72]]
Поріг 0.3: матриця
[[71 22]
 [ 8 79]]
→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)


---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
___
бо її кінцевий результат - це категорія, а не число. Хоча вона розраховує число, наприкінці вона застосовує поріг (0.5) і перетворює числове значення у категорію "так" чи "ні" (1 / 0).

2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
___
predict видає суху відповідь, 1 або 0 (так чи ні), а predict_proba показує наскільки модель впевнена (від 0 до 1).  Для бізнесу це корисно, бо можна працювати з тими, у кого 90%+, а не витрачати ресурси на сумнівних, можна крутити поріг (0.3 / 0.5).

3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
___
Pipeline допомагає автоматизувати масштабування значень одразу для тренувальних даних та тестувальних, нам не потрібно кожного разу прописувати що ми ці дані масштабуємо. Якщо масштабувати дані до розділення на train/test - модель розрахує середнє та розкид по всіх даних, включно з тестовими ознаками і тест перестане бути 'невидимим'. Оцінка вийде завищеною, тож потрібно масштабувати післе розділення на train/test.

4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
___
Логістична регресія складає всі ознаки разом і працює з ними одночасно, тому величина чисел важлива, велика означа задавить маленьку. Тому потрібно масштабування, щоб всі ознаки були в однакових межах.
Дерево рішень працює інакше - воно ставить питання по одній ознаці за раз і дивиться лише на порядок. Масштабування не змінює порядок, тому дереву воно не потрібне.

5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?
___
Від'ємний коефіцієнт означає, що чим більше клієнт звертається в підтримку, тим менший шанс, що він перейде на преміум. Найімовірніше, це незадоволені клієнти з проблемами — їм не варто пропонувати платний апгрейд, спершу треба розв'язати їхні проблеми.

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.